In [1]:
"""
Loan Approval Prediction — PyTorch Feed-Forward Neural Network
----------------------------------------------------------------
Binary classifier predicting loan approval from applicant features.

Dataset: ~20,000-record "Financial Risk for Loan Approval" dataset (Kaggle).
Architecture: input -> 256 -> 128 -> 64 -> 1 (BatchNorm + Dropout, ~51K params)
Result: 94% test accuracy, 0.99 AUC.

Requirements: torch, scikit-learn, pandas, numpy, kagglehub
    pip install torch scikit-learn pandas numpy kagglehub
"""

'\nLoan Approval Prediction — PyTorch Feed-Forward Neural Network\n----------------------------------------------------------------\nBinary classifier predicting loan approval from applicant features.\n \nDataset: ~20,000-record "Financial Risk for Loan Approval" dataset (Kaggle).\nArchitecture: input -> 256 -> 128 -> 64 -> 1 (BatchNorm + Dropout, ~51K params)\nResult: 94% test accuracy, 0.99 AUC.\n \nRequirements: torch, scikit-learn, pandas, numpy, kagglehub\n    pip install torch scikit-learn pandas numpy kagglehub\n'

In [2]:
import os
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score

In [3]:

# ── 1. Load dataset ───────────────────────────────────────────
import kagglehub

path = kagglehub.dataset_download("lorenzozoppelletto/financial-risk-for-loan-approval")
csv_file = [f for f in os.listdir(path) if f.endswith(".csv")][0]
df = pd.read_csv(f"{path}/{csv_file}")
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")

100%|██████████| 2.15M/2.15M [00:00<00:00, 185MB/s]

Extracting files...
Loaded 20000 rows, 36 columns


In [4]:
# ── 2. Preprocessing ──────────────────────────────────────────
TARGET = "LoanApproved"  # binary classification target

# RiskScore is a data-leakage risk for LoanApproved -> drop if present
df = df.drop(columns=["RiskScore"], errors="ignore")

# Encode categorical columns
for col in df.select_dtypes(include="object").columns:
    if col != TARGET:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
if df[TARGET].dtype == object:
    df[TARGET] = LabelEncoder().fit_transform(df[TARGET])

df = df.fillna(df.median(numeric_only=True))

X = df.drop(columns=[TARGET]).values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)
print(f"Features: {X.shape[1]}  |  Class balance: {Counter(y.astype(int))}")

Features: 34  |  Class balance: Counter({np.int64(0): 15220, np.int64(1): 4780})


In [5]:
# ── 3. Train / val / test split + scaling ─────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


def to_loader(X, y, batch_size=256, shuffle=True):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


train_loader = to_loader(X_train, y_train)
val_loader = to_loader(X_val, y_val, shuffle=False)
test_loader = to_loader(X_test, y_test, shuffle=False)

In [6]:
# ── 4. Model ───────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class LoanANN(nn.Module):
    """
    Feed-forward ANN for loan approval classification.
    Architecture: input -> 256 -> 128 -> 64 -> 1
    BatchNorm + Dropout after each hidden layer for regularization.
    """

    def __init__(self, in_features: int, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),  # raw logit -> BCEWithLogitsLoss
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


model = LoanANN(in_features=X_train.shape[1]).to(DEVICE)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

LoanANN(
  (net): Sequential(
    (0): Linear(in_features=34, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=64, out_features=1, bias=True)
  )
)
Parameters: 51,073


In [7]:

# ── 5. Training ───────────────────────────────────────────────
# Handle class imbalance with pos_weight
neg, pos = Counter(y_train.astype(int)).values()
pos_weight = torch.tensor([neg / pos], dtype=torch.float32).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

EPOCHS = 30
best_val = 0.0

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(y_b)
    scheduler.step()
    train_loss = running_loss / len(y_train)

    # ── Validate ───────────────────────────────────────────
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            all_logits.append(model(X_b.to(DEVICE)).cpu())
            all_labels.append(y_b)
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    val_acc = ((all_logits > 0).float() == all_labels).float().mean().item()
    val_auc = roc_auc_score(all_labels.numpy(), torch.sigmoid(all_logits).numpy())

    if val_auc > best_val:
        best_val = val_auc
        torch.save(model.state_dict(), "loan_ann_best.pt")

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {train_loss:.4f} | "
              f"Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}")

print(f"\nBest Val AUC: {best_val:.4f}")

Epoch   1/30 | Loss: 0.5598 | Val Acc: 0.8804 | Val AUC: 0.9867
Epoch   5/30 | Loss: 0.2315 | Val Acc: 0.9137 | Val AUC: 0.9896
Epoch  10/30 | Loss: 0.2100 | Val Acc: 0.9337 | Val AUC: 0.9901
Epoch  15/30 | Loss: 0.1920 | Val Acc: 0.9447 | Val AUC: 0.9904
Epoch  20/30 | Loss: 0.1850 | Val Acc: 0.9247 | Val AUC: 0.9898
Epoch  25/30 | Loss: 0.1771 | Val Acc: 0.9341 | Val AUC: 0.9897
Epoch  30/30 | Loss: 0.1753 | Val Acc: 0.9357 | Val AUC: 0.9897

Best Val AUC: 0.9905


In [8]:
# ── 6. Test evaluation ────────────────────────────────────────
model.load_state_dict(torch.load("loan_ann_best.pt", map_location=DEVICE))
model.eval()

all_logits, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        all_logits.append(model(X_b.to(DEVICE)).cpu())
        all_labels.append(y_b)

all_logits = torch.cat(all_logits)
all_labels = torch.cat(all_labels)
all_preds = (all_logits > 0).float()
test_auc = roc_auc_score(all_labels.numpy(), torch.sigmoid(all_logits).numpy())

print(f"\nTest AUC: {test_auc:.4f}")
print(classification_report(
    all_labels.numpy(), all_preds.numpy(), target_names=["Rejected", "Approved"]
))


Test AUC: 0.9910
              precision    recall  f1-score   support

    Rejected       0.99      0.93      0.96      2283
    Approved       0.81      0.97      0.88       717

    accuracy                           0.94      3000
   macro avg       0.90      0.95      0.92      3000
weighted avg       0.95      0.94      0.94      3000

